# NB04 noise conditions
Own clips mixed with environment noise at SNR 20, 10, 5 dB (fixed seed). Per condition:
Jalur B PER (wav2vec2, same as NB01) and Jalur A WER (zipformer greedy, offline feed, same model as NB03).
Timing is not repeated here; noise does not change model size. See NB01 and NB03 for latency.

Settings: GPU T4 (for wav2vec2), Internet ON. Inputs: `spellspeak-own-clips`, a noise dataset (DEMAND or MUSAN noise subset)
attached as a Kaggle Dataset. Set `NOISE_DIR` below.

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
def cpu_name() -> str:
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def device_info(force_cpu: bool = False) -> dict:
    name, threads = cpu_name(), os.cpu_count()
    try:
        import torch
        threads = torch.get_num_threads()
        if torch.cuda.is_available() and not force_cpu:
            name = torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return {"name": name, "threads": threads, "platform": platform.platform()}


def versions(*pkgs: str) -> dict:
    out = {}
    for p in pkgs:
        try:
            out[p] = metadata.version(p)
        except metadata.PackageNotFoundError:
            out[p] = None
    return out


def write_result(notebook: str, data: dict, metrics: dict, notes: str, pkgs: tuple = ()) -> Path:
    """Shape from docs/NOTEBOOKS.md. Only values computed in this run go into metrics."""
    out = {"notebook": notebook, "date": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "device": device_info(), "data": data, "metrics": metrics, "notes": notes,
           "versions": versions(*pkgs)}
    path = RESULTS / f"{notebook}.json"
    path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out, indent=2, ensure_ascii=False))
    return path

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!pip install -q phonemizer sherpa-onnx jiwer
!wget -q -nc https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2 && tar xjf sherpa-onnx-streaming-zipformer-en-2023-06-26.tar.bz2

In [ ]:
NOISE_DIR = Path("/kaggle/input/demand")   # any folder of noise .wav files
SNRS_DB = [20, 10, 5]
SEED = 0

In [ ]:
# Own recordings: private Kaggle Dataset made from data/own (backend/record_clips.py output):
#   metadata.csv + wav/*.wav, 16 kHz mono. Real consenting speakers only. Never synthetic clips.
import numpy as np
import pandas as pd
import soundfile as sf

OWN = Path("/kaggle/input/spellspeak-own-clips")
if not (OWN / "metadata.csv").exists():
    raise FileNotFoundError(f"{OWN}/metadata.csv not found. Record clips with backend/record_clips.py, "
                            "upload data/own as a private Kaggle Dataset, add it to this notebook.")
meta = pd.read_csv(OWN / "metadata.csv")
assert meta["consent"].astype(str).str.lower().eq("yes").all(), "every row needs consent=yes"


def load_wav(path: Path) -> np.ndarray:
    audio, sr = sf.read(path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        raise ValueError(f"{path}: need 16000 Hz, got {sr}")
    return audio


own_items = [{"id": r.file, "speaker": r.speaker_id, "word": r.word, "audio": load_wav(OWN / r.file)}
             for r in meta.itertuples()]
print(f"own clips: {len(own_items)} | speakers: {meta.speaker_id.nunique()} | words: {meta.word.nunique()}")

In [ ]:
from phonemizer import phonemize
from phonemizer.separator import Separator

_ipa_cache: dict[str, list[str]] = {}


def target_ipa(word: str) -> list[str]:
    """Same call as backend/server.py so the notebook scores exactly what the app scores."""
    if word not in _ipa_cache:
        out = phonemize(word, language="en-us", backend="espeak",
                        separator=Separator(phone=" ", word="", syllable=""), strip=True)
        _ipa_cache[word] = out.split()
    return _ipa_cache[word]

In [ ]:
import random
import torch
import torchaudio


def mix_snr(clean: np.ndarray, noise: np.ndarray, snr_db: float) -> np.ndarray:
    """Scale noise so 10*log10(P_clean / P_noise) == snr_db, then add."""
    p_clean = np.mean(clean ** 2)
    p_noise = np.mean(noise ** 2)
    scale = np.sqrt(p_clean / (p_noise * 10 ** (snr_db / 10)))
    return (clean + scale * noise).astype(np.float32)


noise_files = sorted(NOISE_DIR.rglob("*.wav"))
if not noise_files:
    raise FileNotFoundError(f"no .wav files under {NOISE_DIR}. Attach a noise dataset and set NOISE_DIR.")
_noise_cache: dict[Path, np.ndarray] = {}


def noise_audio(path: Path) -> np.ndarray:
    if path not in _noise_cache:
        wav, sr = torchaudio.load(str(path))
        _noise_cache[path] = torchaudio.functional.resample(wav.mean(0), sr, 16000).numpy()
    return _noise_cache[path]


def noisy_versions(items: list[dict]) -> dict[str, list[dict]]:
    rng = random.Random(SEED)
    out = {"clean": items}
    for snr in SNRS_DB:
        out[f"snr{snr}"] = []
    for it in items:  # one noise file and offset per clip, shared across SNRs
        nf = rng.choice(noise_files)
        n = noise_audio(nf)
        L = len(it["audio"])
        if len(n) < L:
            n = np.tile(n, L // len(n) + 1)
        off = rng.randrange(0, len(n) - L + 1)
        seg = n[off:off + L]
        for snr in SNRS_DB:
            out[f"snr{snr}"].append({**it, "audio": mix_snr(it["audio"], seg, snr), "noise": f"{nf.name}@{off}"})
    return out


conditions = noisy_versions(own_items)
print({k: len(v) for k, v in conditions.items()}, "| noise files:", len(noise_files))

In [ ]:
import jiwer
import re
import sherpa_onnx
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import per as per_mod

MODEL = "facebook/wav2vec2-lv-60-espeak-cv-ft"
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = Wav2Vec2Processor.from_pretrained(MODEL)
w2v = Wav2Vec2ForCTC.from_pretrained(MODEL).eval().to(dev)

ZDIR = Path("sherpa-onnx-streaming-zipformer-en-2023-06-26")


def pick(pattern: str) -> str:
    hits = sorted(ZDIR.glob(pattern))
    return str(([h for h in hits if ".int8." in h.name] or hits)[0])


zrec = sherpa_onnx.OnlineRecognizer.from_transducer(
    tokens=str(ZDIR / "tokens.txt"), encoder=pick("encoder-*.onnx"), decoder=pick("decoder-*.onnx"),
    joiner=pick("joiner-*.onnx"), num_threads=1, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")


def phones(audio: np.ndarray) -> list[str]:
    x = processor(audio, sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    with torch.inference_mode():
        ids = torch.argmax(w2v(x).logits, dim=-1).cpu()
    return processor.batch_decode(ids)[0].split()


def asr(audio: np.ndarray) -> str:
    s = zrec.create_stream()
    s.accept_waveform(16000, audio)
    s.input_finished()
    while zrec.is_ready(s):
        zrec.decode_stream(s)
    r = zrec.get_result(s)
    return r if isinstance(r, str) else r.text


def norm(text: str) -> str:
    return " ".join(re.sub(r"[^a-z' ]", " ", text.lower()).split())

In [ ]:
rows = []
for cond, items in conditions.items():
    for it in items:
        ref = target_ipa(it["word"])
        r = per_mod.per(ref, phones(it["audio"]))
        rows.append({"condition": cond, "id": it["id"], "speaker": it["speaker"], "noise": it.get("noise"),
                     "S": r["S"], "D": r["D"], "I": r["I"], "N": r["N"], "per": r["per"],
                     "ref_text": norm(it["word"]), "hyp_text": norm(asr(it["audio"]))})
noise_df = pd.DataFrame(rows)
noise_df.to_csv(RESULTS / "NB04_per_clip.csv", index=False)

metrics = {}
for cond, g in noise_df.groupby("condition", sort=False):
    metrics[cond] = {"n": len(g), "per_corpus": int((g.S + g.D + g.I).sum()) / int(g.N.sum()),
                     "wer_zipformer": jiwer.wer(g.ref_text.tolist(), [h or "<empty>" for h in g.hyp_text.tolist()])}
write_result(
    "NB04_noise_conditions",
    data={"source": f"own clips + noise from {NOISE_DIR.name}", "n_utterances": len(own_items),
          "speakers": int(meta.speaker_id.nunique()), "noise_files": len(noise_files)},
    metrics=metrics,
    notes=f"SNR {SNRS_DB} dB, seed={SEED}, one random noise file and offset per clip, shared across SNRs. "
          "PER: wav2vec2 greedy CTC. WER: zipformer greedy, whole clip fed at once (no real-time pacing).",
    pkgs=("torch", "torchaudio", "transformers", "phonemizer", "sherpa-onnx", "jiwer"),
)